In [1]:
# 导入LangChain的字符切分器
from langchain_text_splitters import CharacterTextSplitter

# 准备一段示例文本
text = "人工智能（Artificial Intelligence），英文缩写为AI。它是研究、开发用于模拟、延伸和扩展人的智能的理论、方法、技术及应用系统的一门新的技术科学。AI是计算机科学的一个分支，它企图了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器。"

# 初始化切分器，设置块大小和重叠大小
text_splitter = CharacterTextSplitter(
    separator="。",  # 尝试以句号作为分隔符，但如果块依然过长，会强制切分
    chunk_size=50,
    chunk_overlap=10,
    length_function=len,
    is_separator_regex=False,
)

# 执行切分
chunks = text_splitter.create_documents([text])

# 打印结果
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print(f"Length: {len(chunk.page_content)}\n")

--- Chunk 1 ---
人工智能（Artificial Intelligence），英文缩写为AI
Length: 37

--- Chunk 2 ---
它是研究、开发用于模拟、延伸和扩展人的智能的理论、方法、技术及应用系统的一门新的技术科学
Length: 44

--- Chunk 3 ---
AI是计算机科学的一个分支，它企图了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器
Length: 53



In [2]:
from langchain_text_splitters import TokenTextSplitter

text = (
    "LangChain is a framework for developing applications powered by large language models (LLMs). "
    "It simplifies the entire application lifecycle."
)

splitter = TokenTextSplitter(
    encoding_name="cl100k_base", chunk_size=20, chunk_overlap=5
)

chunks = splitter.split_text(text)
for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---\n{c}\n")

--- Chunk 1 ---
LangChain is a framework for developing applications powered by large language models (LLMs). It simplifies

--- Chunk 2 ---
Ms). It simplifies the entire application lifecycle.



In [3]:
# 导入递归字符切分器
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 准备包含长短句的文本
text = "RAG is powerful. It combines retrieval and generation. But how to choose the best chunking strategy? This is a critical question for all developers working on RAG systems."

# 初始化切分器，将句子分隔符放在最前面
sentence_splitter = RecursiveCharacterTextSplitter(
    separators=[". ", "? ", "! ", "\n\n", "\n", " "],  # 优先使用句子结束符
    chunk_size=80,  # 设置一个合理的块大小
    chunk_overlap=10,
    keep_separator=True,  # 保留分隔符
)

# 执行切分
chunks = sentence_splitter.create_documents([text])

# 打印结果
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("")

--- Chunk 1 ---
RAG is powerful. It combines retrieval and generation

--- Chunk 2 ---
. But how to choose the best chunking strategy

--- Chunk 3 ---
? This is a critical question for all developers working on RAG systems.



In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 一段包含段落、换行和长句的复杂文本
long_text = """
第一章：RAG的崛起

检索增强生成（RAG）彻底改变了我们与大型语言模型的交互方式。它通过动态地从外部知识源中检索信息来增强模型的响应能力。

这项技术解决了两个核心问题：
1. 知识时效性：模型不再局限于其训练数据中的静态知识。
2. 可靠性：通过引用来源，减少了模型产生“幻觉”的风险。

然而，RAG的成功在很大程度上取决于其检索组件的质量，而检索质量又直接受到文档切分策略的影响。一个糟糕的切分会毁掉整个系统。
"""

# 初始化递归字符切分器
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,  # 添加块在原文中的起始位置元数据
)

# 执行切分
chunks = recursive_splitter.create_documents([long_text])

# 打印结果
for chunk in chunks:
    print(f"--- Chunk (Start Index: {chunk.metadata['start_index']}) ---")
    print(chunk.page_content)
    print("")

--- Chunk (Start Index: 1) ---
第一章：RAG的崛起

检索增强生成（RAG）彻底改变了我们与大型语言模型的交互方式。它通过动态地从外部知识源中检索信息来增强模型的响应能力。

--- Chunk (Start Index: 74) ---
这项技术解决了两个核心问题：
1. 知识时效性：模型不再局限于其训练数据中的静态知识。
2. 可靠性：通过引用来源，减少了模型产生“幻觉”的风险。

--- Chunk (Start Index: 149) ---
然而，RAG的成功在很大程度上取决于其检索组件的质量，而检索质量又直接受到文档切分策略的影响。一个糟糕的切分会毁掉整个系统。



In [5]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_string = """
<!DOCTYPE html>
<html>
<body>
<h1>RAG系统架构</h1>
<p>RAG系统主要由索引和检索生成两部分组成。</p>
<h2>索引流程</h2>
<p>索引流程包括加载、切分、嵌入和存储。</p>
<h3>切分策略</h3>
<p>切分是索引流程中的关键步骤。</p>
<h2>检索流程</h2>
<p>检索流程负责根据用户查询找到相关文档块。</p>
</body>
</html>
"""

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_header_splits = html_splitter.split_text(html_string)

for split in html_header_splits:
    print(split)
    print("-" * 20)

page_content='RAG系统架构' metadata={'Header 1': 'RAG系统架构'}
--------------------
page_content='RAG系统主要由索引和检索生成两部分组成。' metadata={'Header 1': 'RAG系统架构'}
--------------------
page_content='索引流程' metadata={'Header 1': 'RAG系统架构', 'Header 2': '索引流程'}
--------------------
page_content='索引流程包括加载、切分、嵌入和存储。' metadata={'Header 1': 'RAG系统架构', 'Header 2': '索引流程'}
--------------------
page_content='切分策略' metadata={'Header 1': 'RAG系统架构', 'Header 2': '索引流程', 'Header 3': '切分策略'}
--------------------
page_content='切分是索引流程中的关键步骤。' metadata={'Header 1': 'RAG系统架构', 'Header 2': '索引流程', 'Header 3': '切分策略'}
--------------------
page_content='检索流程' metadata={'Header 1': 'RAG系统架构', 'Header 2': '检索流程'}
--------------------
page_content='检索流程负责根据用户查询找到相关文档块。' metadata={'Header 1': 'RAG系统架构', 'Header 2': '检索流程'}
--------------------


In [6]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

PYTHON_CODE = """
class RAGSystem:
    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator

    def query(self, question: str) -> str:
        \"\"\"Processes a query and returns an answer.\"\"\"
        context = self.retriever.retrieve(question)
        answer = self.generator.generate(question, context)
        return answer

def setup_retriever():
    # Sets up the retriever component
    print("Retriever setup complete.")
    return True
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=200, chunk_overlap=0
)
python_docs = python_splitter.create_documents([PYTHON_CODE])

for doc in python_docs:
    print(doc.page_content)
    print("-" * 20)

class RAGSystem:
    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator
--------------------
def query(self, question: str) -> str:
        """Processes a query and returns an answer."""
        context = self.retriever.retrieve(question)
--------------------
answer = self.generator.generate(question, context)
        return answer
--------------------
def setup_retriever():
    # Sets up the retriever component
    print("Retriever setup complete.")
    return True
--------------------


In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embeddings = OpenAIEmbeddings(
    model=os.getenv("EMBEDDING_MODEL"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    sentence_split_regex=r"(?<=[。！？.!?])",  # 让中文能按句子切
)

semantic_text = "自动驾驶汽车利用传感器和人工智能在无需人类干预的情况下导航。 其核心技术包括计算机视觉、激光雷达和GPS。这项技术有望彻底改变交通运输行业。 另一方面，量子计算是一种利用量子力学原理进行计算的新兴技术。 量子比特或qubit是其基本单位，可以同时存在于多种状态，从而实现强大的并行计算能力。"

docs = text_splitter.create_documents([semantic_text])
print(docs)

[Document(metadata={}, page_content='自动驾驶汽车利用传感器和人工智能在无需人类干预的情况下导航。  其核心技术包括计算机视觉、激光雷达和GPS。'), Document(metadata={}, page_content='这项技术有望彻底改变交通运输行业。  另一方面，量子计算是一种利用量子力学原理进行计算的新兴技术。  量子比特或qubit是其基本单位，可以同时存在于多种状态，从而实现强大的并行计算能力。 ')]


In [9]:
import os

from datasets import Dataset

# 1) 准备测试数据集
test_data = {
    "question": [
        "What is Recursive Character Splitting?",
        "What is the main advantage of Semantic Chunking?",
    ],
    "contexts": [
        [
            "Recursive Character Splitting tries to split based on a list of separators.",
            "It is the recommended method for generic text.",
        ],
        [
            "Semantic Chunking splits text based on semantic meaning, using embedding similarity.",
            "This creates highly coherent chunks.",
        ],
    ],
    "answer": [
        "Recursive Character Splitting is a method that uses a list of separators to split text and is recommended for general use.",
        "The main advantage of Semantic Chunking is that it creates chunks that are semantically very coherent.",
    ],
    "ground_truth": [
        "Recursive Character Text Splitting is a text splitting method that recursively tries to split text using a list of separators like newlines.",
        "The primary benefit of Semantic Chunking is its ability to create semantically cohesive blocks of text, ensuring that related ideas are kept together.",
    ],
}
dataset = Dataset.from_dict(test_data)

# 3) 配置 LLM 和 Embeddings
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

# 2) 导入 RAGAS 与指标（注意，即将 deprecated，替换为 ragas.metrics.collections 则不能使用 evaluate 旧接口）
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

# 3) 配置 LLM 和 Embeddings
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("OPENAI_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")

chat_llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,
    temperature=0.0,  # 评估更稳定
)

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,
)

# 4) 运行评估（注意：某些度量需要 embeddings；稳妥起见一并传入）
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
    llm=chat_llm,
    embeddings=embeddings,
)

# 5) 打印结果
print("Ragas评估结果：")
print(result)  # 汇总分

# 明细（逐条样本）
df = result.to_pandas()
print("\n逐样本得分：")
print(df.head())

/tmp/ipykernel_1981567/583800696.py:38: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_1981567/583800696.py:38: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_1981567/583800696.py:38: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_1981567/583800696.py:38: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and w

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Ragas评估结果：
{'faithfulness': 0.5000, 'answer_relevancy': 1.0000, 'context_recall': 0.5000, 'context_precision': 1.0000}

逐样本得分：
                                         user_input  \
0            What is Recursive Character Splitting?   
1  What is the main advantage of Semantic Chunking?   

                                  retrieved_contexts  \
0  [Recursive Character Splitting tries to split ...   
1  [Semantic Chunking splits text based on semant...   

                                            response  \
0  Recursive Character Splitting is a method that...   
1  The main advantage of Semantic Chunking is tha...   

                                           reference  faithfulness  \
0  Recursive Character Text Splitting is a text s...           1.0   
1  The primary benefit of Semantic Chunking is it...           0.0   

   answer_relevancy  context_recall  context_precision  
0               1.0             0.0                1.0  
1               1.0             1.0         